# Ingestão na Camada Bronze com PyAirbyte

**Disciplina:** Engenharia de Dados — Ingestão de Dados e a Camada Bronze

Neste notebook vamos extrair dados de uma fonte usando um **conector do Airbyte** e gravá-los, sem transformação, na **camada Bronze** do nosso data lake (Silo/MinIO).

### Por que PyAirbyte e não o Airbyte completo?
O Airbyte com interface web (instalado via `abctl`) roda sobre Kubernetes e exige, na prática, **16 GB de RAM** na máquina. O **PyAirbyte** é a biblioteca Python oficial do Airbyte: usa **os mesmos conectores** (sources), mas roda direto no Python, sem Kubernetes — funciona bem em notebooks de 8 GB.

| Conceito do Airbyte | No PyAirbyte |
|---|---|
| Source (conector de origem) | `ab.get_source(...)` |
| Streams (tabelas da fonte) | `source.select_streams(...)` |
| Sync | `source.read()` |
| Destination | nós mesmos gravamos no bucket `bronze` |

### Pré-requisitos
1. Ambiente subido com `docker compose -f compose.yml up -d` (serviços `minio` e `spark-jupyter`).
2. Este notebook aberto no Jupyter do container (http://localhost:8888).
3. Bucket `bronze` criado com `python scripts/2_setup_minio.py` no ambiente local.


## 1. Instalação das bibliotecas
Execute uma única vez por container. Leva cerca de 1 minuto.

In [ ]:
%pip install -q airbyte s3fs

## 2. Configuração de acesso ao data lake
As credenciais já vêm das variáveis de ambiente definidas no `compose.yml`. Os valores após a vírgula são usados caso as variáveis não existam.

In [ ]:
import os
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
import s3fs

S3_ENDPOINT = os.getenv("AWS_ENDPOINT_URL", "http://minio:9000")
S3_KEY      = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
S3_SECRET   = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin123")

# Opções usadas pelo pandas para ler/gravar em s3://
storage_options = {
    "key": S3_KEY,
    "secret": S3_SECRET,
    "client_kwargs": {"endpoint_url": S3_ENDPOINT},
}
fs = s3fs.S3FileSystem(**storage_options)

BUCKET_BRONZE = "bronze"
if not fs.exists(BUCKET_BRONZE):
    fs.mkdir(BUCKET_BRONZE)

print("Endpoint:", S3_ENDPOINT)
print("Buckets disponíveis:", fs.ls(""))

## 3. Configurando o conector de origem (Source)
Usaremos o **`source-faker`**, um conector do Airbyte que gera dados fictícios de um e-commerce (usuários, produtos e compras). Ele é ótimo para estudo porque não exige credenciais.

Na primeira execução o PyAirbyte instala o conector em um ambiente Python isolado (pode levar 1–2 minutos).

- `count`: quantidade de registros de usuários/compras gerados (mantenha valores pequenos em máquinas com pouca memória).
- `seed`: semente para gerar sempre os mesmos dados (reprodutibilidade).

In [ ]:
import airbyte as ab

source = ab.get_source(
    "source-faker",
    config={"count": 1000, "seed": 42},
    install_if_missing=True,
)

source.check()   # testa a conexão/configuração, como o botão "Test" da interface do Airbyte
print("Streams disponíveis:", source.get_available_streams())

## 4. Extração (sync)
Selecionamos os streams desejados e executamos a leitura. Os registros ficam temporariamente em um **cache local DuckDB** gerenciado pelo PyAirbyte — é a área de staging antes de gravarmos no data lake.

In [ ]:
source.select_streams(["users", "products", "purchases"])
result = source.read()

for nome, dataset in result.streams.items():
    print(f"{nome:10s} -> {len(dataset.to_pandas()):>6} registros")

Vamos espiar os dados extraídos. Repare nas colunas **`_airbyte_raw_id`**, **`_airbyte_extracted_at`** e **`_airbyte_meta`**: são metadados de linhagem adicionados pelo Airbyte.

In [ ]:
df_users = result["users"].to_pandas()
df_users.head()

## 5. Gravação na camada Bronze
Princípios da camada Bronze aplicados aqui:

1. **Dado cru, sem transformação** — gravamos exatamente o que veio da fonte.
2. **Metadados de ingestão** — acrescentamos `_ingestion_ts` (quando foi ingerido) e `_source` (de onde veio).
3. **Append-only** — cada execução gera um **novo arquivo**; nada é sobrescrito, o que permite reprocessar o histórico.
4. **Particionamento por data de ingestão** — `ingestion_date=AAAA-MM-DD`, padrão reconhecido por Spark, pandas e DuckDB.
5. **Formato colunar** — Parquet.

Estrutura resultante:
```
bronze/
└── faker/
    ├── users/ingestion_date=2026-10-11/users_143015.parquet
    ├── products/ingestion_date=2026-10-11/products_143015.parquet
    └── purchases/ingestion_date=2026-10-11/purchases_143015.parquet
```

In [ ]:
FONTE = "faker"

def gravar_bronze(df: pd.DataFrame, stream: str, ingestao: datetime) -> str:
    """Grava um DataFrame cru na camada Bronze, particionado por data de ingestão."""
    df = df.copy()
    df["_ingestion_ts"] = ingestao
    df["_source"] = f"source-{FONTE}"

    caminho = (
        f"s3://{BUCKET_BRONZE}/{FONTE}/{stream}/"
        f"ingestion_date={ingestao:%Y-%m-%d}/{stream}_{ingestao:%H%M%S}_{uuid4().hex}.parquet"
    )
    df.to_parquet(
        caminho,
        index=False,
        storage_options=storage_options,
        coerce_timestamps="us",          # microssegundos: formato aceito pelo Spark
        allow_truncated_timestamps=True,
    )
    return caminho

ingestao = datetime.now(timezone.utc)
for nome, dataset in result.streams.items():
    caminho = gravar_bronze(dataset.to_pandas(), nome, ingestao)
    print("Gravado:", caminho)

## 6. Conferindo o resultado
Listamos os arquivos no bucket e lemos de volta uma das tabelas. Você também pode conferir pelo console web em http://localhost:9001.

In [ ]:
for arquivo in fs.find(f"{BUCKET_BRONZE}/{FONTE}"):
    print(arquivo)

In [ ]:
# Lê TODAS as partições do stream users (todas as execuções acumuladas)
df_bronze_users = pd.read_parquet(f"s3://{BUCKET_BRONZE}/{FONTE}/users", storage_options=storage_options)
print(df_bronze_users.shape)
df_bronze_users[["id", "name", "email", "_airbyte_extracted_at", "_ingestion_ts", "ingestion_date"]].head()

## 7. (Opcional) Lendo a Bronze com Spark
O container `spark-jupyter` já está configurado (via `PYSPARK_SUBMIT_ARGS` no compose) para acessar o data lake pelo protocolo `s3a://`. Na primeira execução o Spark baixa o conector `hadoop-aws`, o que pode levar alguns minutos.

Esta é a ponte para a próxima etapa: ler a Bronze e transformar os dados para a camada **Silver**.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("bronze-pyairbyte").getOrCreate()

df_spark = spark.read.parquet(f"s3a://{BUCKET_BRONZE}/{FONTE}/purchases")
df_spark.printSchema()
df_spark.groupBy("ingestion_date").count().show()

## 8. Exercícios
1. **Append-only na prática:** execute novamente as seções 4 e 5. Quantos arquivos existem agora em cada stream? Quantos registros a leitura da seção 6 retorna? O que isso implica para a camada Silver (dica: deduplicação por `id`)?
2. **Outros conectores:** liste os conectores disponíveis com `ab.get_available_connectors()` e escolha um que não exija credenciais (por exemplo, `source-pokeapi` com `config={"pokemon_name": "pikachu"}`). Adapte a variável `FONTE` e grave na Bronze.
3. **Reflexão:** que metadados adicionais você incluiria na Bronze de um ambiente de produção (ex.: versão do conector, identificador da execução, hash do arquivo)?

## Problemas comuns
| Sintoma | Solução |
|---|---|
| `Failed to connect to the connector registry` | Sem acesso à internet/registro. Defina `os.environ["AIRBYTE_OFFLINE_MODE"]="1"` e use `ab.get_source("source-faker", config=..., pip_url="airbyte-source-faker")`. |
| Container lento ou travando | Reduza `count` (ex.: 200) e feche outros programas. |
| `NoSuchBucket` | Crie o bucket `bronze` pelo console em http://localhost:9001. |
| Erro de conexão em `http://minio:9000` | Confirme que o serviço `minio` está *healthy*: `docker compose ps`. |
